# Batch Tax Validation

This notebook is the Jupyter version of the Batch tester app. It reads an Excel input workbook, calls the Valitax by Banqup API service (supports UAT/CVE environments), compares expected values with engine output, and writes an Excel result workbook.

## Before you start

Keep these files together in one folder:

- `batch_tax_validation.ipynb`
- `validation_engine.py`
- `upload_template.xlsx`
- `json_mapper.xlsx`
- `json_mapper_output.xlsx`
- `requirements-notebook.txt`

Open Jupyter from that same folder, then open this notebook. The kernel's current working directory should be the folder containing this notebook and the files above.

The upload template file can be renamed and stored elsewhere but its path needs to be configured in a cell below. There is a limit of 20k lines.
The following operators are supported for result comparisons:
- `\equal` (default, can be omitted)
- `\not_equal`
- `\is_null`
- `\is_not_null`
- `\contains`
- `\not_contains`

If dependencies are missing, run this once in a notebook cell and restart the kernel:

```python
%pip install -r requirements-notebook.txt
```


## 1. Setup

Run this cell first. It checks that Jupyter is running from the folder containing the notebook support files.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, Markdown, display

WORKING_DIR = Path.cwd().resolve()

required_files = [
    "validation_engine.py",
    "json_mapper.xlsx",
    "json_mapper_output.xlsx",
]
missing_files = [name for name in required_files if not (WORKING_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Jupyter is not running from the notebook folder, or files are missing. "
        f"Current folder: {WORKING_DIR}. Missing: {missing_files}. "
        "Start Jupyter from the folder containing this notebook and the support files."
    )

if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))

from validation_engine import run_batch_validation

print(f"Working folder: {WORKING_DIR}")
print("Setup complete.")


## 2. Configuration

Edit only this cell for a normal run.

- `INPUT_XLSX`: your completed input workbook. You can use `upload_template.xlsx` as the starting template.
- `ENVIRONMENT`: `"uat"` or `"cve"`.
- `SPACE_ID`: the Banqup space ID to use for the API calls.
- `PROCESS_MODE`: `"comparisons_only"`, `"full_output"`, or `"both"`.
- Insert OAuth credentials as `OAUTH_CLIENT_ID` and `OAUTH_CLIENT_SECRET`.


In [ ]:
INPUT_XLSX = WORKING_DIR / "upload_template.xlsx" # Renmame this file to your input file name stored within the same folder, or change the path to your input file
ENVIRONMENT = "uat"                                # "uat" or "cve"
SPACE_ID = ""  # Required Banqup space ID
PROCESS_MODE = "both"                              # "comparisons_only", "full_output", or "both"
OAUTH_CLIENT_ID = ""
OAUTH_CLIENT_SECRET = ""

# Optional
OUTPUT_DIR = WORKING_DIR / "outputs"

# Pre-check input file
input_path = Path(INPUT_XLSX).expanduser().resolve()
if not input_path.exists():
    raise FileNotFoundError(f"Input workbook not found: {input_path}")
if input_path.suffix.lower() != ".xlsx":
    raise ValueError("INPUT_XLSX must point to an .xlsx file.")


## 3. Run validation

This cell makes the API calls and writes the result workbook into the `outputs` folder. Depending on the number of input rows, this may take some time.


In [ ]:
LAST_RESULT = run_batch_validation(INPUT_XLSX, environment=ENVIRONMENT, space_id=SPACE_ID, process_mode=PROCESS_MODE, output_dir=OUTPUT_DIR,
    client_id=OAUTH_CLIENT_ID, client_secret=OAUTH_CLIENT_SECRET,
)
display(Markdown("✅ **Validation completed.**"))
output_path = LAST_RESULT["output_path"]
stats = LAST_RESULT["stats"]
stats_table = pd.DataFrame([{"Metric": key.replace("_", " ").title(), "Value": round(value, 2)} for key, value in stats.items()])
display(stats_table)
display(FileLink(str(output_path), result_html_prefix="Download result: "))